## Part B - Stoichiometry and reaction balancing (30 points)

1. **Reaction Balancer Function (20 points)**
   - Write a function `balance_reaction(reactants, products)` that:
     - Takes as input:
       - `reactants`: list of chemical formulas for reactants (e.g., `["H2", "O2"]`)
       - `products`: list of chemical formulas for products (e.g., `["H2O"]`)
     - Returns: a **list of stoichiometric coefficients** that balances the reaction.
     - You don't need to consider parenthesis or "." in the formula for this problem.
     - **Hint:** You can represent the reaction as a system of linear equations based on the number of atoms of each element. Look up the term Singular Value Decomposition, you can use [this](https://en.wikipedia.org/wiki/Singular_value_decomposition) link.
     - Test it for 3 reactions of your choice.


In [13]:
import numpy as np
from fractions import Fraction
from math import gcd


def parse_elements(compoundlist:list[str])->set[str]:
    """
    Returns a set of elements present in the current list of atoms 
    """
    elements = set()
    for reactant in compoundlist:
            for index, character in enumerate(reactant):
                if character.isupper():
                    element = character
                    if index+1 < len(reactant) and reactant[index+1].islower():  ### To account for two symbol elements
                        element+=reactant[index+1]
                    elements.add(element)
    return elements

def read_number(atom:str, index:int) -> tuple[int,int]:
    """
    Reads the digits starting at index untill the next non-digit. Returns (number, index just after them), no digits meaning 1.
    """
    digits = ""
    while index < len(atom) and atom[index].isdigit():
        digits += atom[index]
        index += 1
    return int(digits) if digits else 1, index


## This code (no ai) works if we don't account for nesting like brackets or parantheses. 
def numberofatoms_no_nesting(atom:str) -> dict[str,int]:
    """
    Returns a dict with atoms as keys and number of times this atom occurs in the molecule as value
    """
    mydict:dict[str,int]= {}
    elements = parse_elements([atom])
    for element in elements:
        counter:int = 0
        for index, character in enumerate(atom):

            if atom[index:index+len(element)] == element:

                digit_index = index + len(element)                                                                                          # start looking for numbers right after the element's letters
                if len(element) == 1 and digit_index < len(atom) and atom[digit_index].islower():                                           #prevents 1 character elements misfiring when they share the same letter with a 2 letter element like Cl (good thing i noticed that, tough stuff)
                    continue
                digits = ""                                                                                                                 #Starts collecting digits into a string, needs to be converted to int later ?
                while digit_index < len(atom) and atom[digit_index].isdigit():
                    digits += atom[digit_index]                                                                                             # keep grabbing characters as long as they're digits
                    digit_index += 1

                counter += int(digits) if digits else 1                                                                                     # no digits found = implicit count of 1

        mydict[element] = counter
    return mydict

### This code DOES account for nesting, but uses a stack structure proposed by AI. It works great but the asignment said we can ommit paranthesis.
def numberofatoms_with_nesting(atom:str) -> dict[str,int]:
    """
    Returns a dict with atoms as keys and number of times this atom occurs in the molecule as value.
    Parentheses and brackets are handled by keeping one dict per nesting level on a stack: opening a
    group pushes a fresh dict, closing it pops that dict and merges it into the level below, each
    count multiplied by the group's subscript.
    """
    stack:list[dict[str,int]] = [{}]
    index:int = 0

    while index < len(atom):
        character = atom[index]

        if character == "(" or character == "[":
            stack.append({})
            index += 1

        elif character == ")" or character == "]":
            group = stack.pop()
            multiplier, index = read_number(atom, index + 1)                              #Multiplies the 
            for element, count in group.items():
                stack[-1][element] = stack[-1].get(element, 0) + count * multiplier

        elif character.isupper():
            element = character
            index += 1
            if index < len(atom) and atom[index].islower():                 ### To account for two symbol elements
                element += atom[index]
                index += 1
            count, index = read_number(atom, index)
            stack[-1][element] = stack[-1].get(element, 0) + count

        else:
            index += 1                                                      # anything else just gets skipped

    return stack[0]


def balance_reaction(reactants:list[str], products:list[str])->list[str]:  
    """Takes in reactants and products as a list of strings and returns a list of stoechiometric coefficients corresponding to them"""
    numberofatoms = numberofatoms_with_nesting if nesting else numberofatoms_no_nesting
    ###Staging the matrix                                                                                                            
    elements = list[str](parse_elements(reactants) | parse_elements(products))                                                          #Converting the elemets set to list since we will iterate over them later 
    rows:int = len(elements)                                                                                                            #will represent elements in the matrix
    columns:int = len(reactants)+len(products)                                                                                          #represent stoechiometric coefficients in the matrix
    my_matrix = np.zeros((rows,columns))
    compounds = reactants + products

    ###Forming the matrix to solve
    for row in range(rows):
        for column in range(columns):
            if column<len(reactants):
                my_matrix[row, column] = numberofatoms(compounds[column]).get(elements[row], 0)
            else:
                my_matrix[row, column] = -numberofatoms(compounds[column]).get(elements[row], 0)
    stoechiometric_coefficients:list[int] = solve_matrix(my_matrix)

    ###Debug, displays nicely how the matrix to solve with SVD was formed
    if debug:
        
        print("Rows and columns represent:\n", "   ",*reactants,"->", *products)
        print(*elements, sep="\n")
        print(my_matrix)

    return stoechiometric_coefficients


def solve_matrix(matrix:np.ndarray) -> list[int]:
    """
    Takes the m x n matrix built by balance_reaction and returns the smallest vector of
    whole numbers x such that matrix @ x = 0.

    The columns of V (from the SVD matrix = U @ S @ V.T) that pair with a zero singular
    value span the null space, so the last one of them is a valid solution. It comes out
    as floats, so it gets rescaled into the smallest equivalent integers.
    """
    U, singular_values, Vt = np.linalg.svd(matrix)

    tolerance = max(matrix.shape) * np.finfo(float).eps * singular_values[0]                     # a singular value is never exactly 0 in floating point
    rank:int = int(np.sum(singular_values > tolerance))
    if rank == matrix.shape[1]:
        raise ValueError("Only the trivial all-zero solution exists, the reaction cannot be balanced")

    solution = Vt[-1]                                                                            # last row of V.T = last column of V = a null space vector

    ###Turning the floats into the smallest possible whole numbers
    solution = solution / np.min(np.abs(solution[np.abs(solution) > 1e-10]))                     # smallest non zero entry becomes 1
    fractions = [Fraction(value).limit_denominator(1000) for value in solution]
    common_denominator:int = 1
    for fraction in fractions:
        common_denominator = common_denominator * fraction.denominator // gcd(common_denominator, fraction.denominator)

    return [int(fraction * common_denominator) for fraction in fractions]


### Tests
debug = False
nesting = False
# test_reactants: list = ["[Cr(N2H4CO)6]4[Cr(CN)6]3", "KMnO4", "H2SO4"]
# test_products:  list = ["K2Cr2O7", "MnSO4", "CO2", "KNO3", "K2SO4", "H2O"]

test_reactants: list = ["C6H12O6", "O2"]
test_products:  list = ["CO2", "H2O"]
balance_reaction(test_reactants,test_products)

[1, 6, 6, 6]

2. **Mass Conservation Check (10 points)**
   - Using the molecular mass calculator from **Part A**, compute the total mass of reactants and products with the coefficients found. Achieve this by writing a function which takes a dictionary of molecules and their coefficient and return a True or False.
   - Verify that the total mass is conserved. Print a message indicating whether the reaction is balanced.
